# **BUSINESS QUESTIONS**

* Kategori produk mana yang paling rentan shrinkage berdasarkan pola void transaction?
* Apakah ada pola waktu (shift, hari) yang korelasi dengan anomali transaksi?
* Kasir atau toko mana yang memiliki void/refund rate di luar batas normal?
* Berapa estimasi kerugian tahunan dan kategori mana prioritas intervensi?
Yang kamu bangun:
* Python: anomaly detection (Isolation Forest) pada data transaksi
* SQL: query void rate, refund pattern, transaksi di luar jam normal
* Power BI: Loss Prevention Dashboard dengan alert threshold merah/kuning/hijau



SHRINKAGE

├── 1. External Theft     → pelanggan mencuri (susah dideteksi dari transaksi)

├── 2. Internal Theft     → karyawan/kasir curang ← BISA dideteksi dari pola void!

└── 3. Administrative     → salah input harga, expired tidak tercatat ← dari anomali data

Dataset Dunnhumby hanya punya data transaksi — jadi kita fokus deteksi Internal Theft dan Administrative Error yang jejaknya ada di data.

# Load Dataset

In [1]:
import os
os.chdir('..')  # naik ke root folder
print(os.getcwd())  # verifikasi lokasi sekarang

d:\Project Portofolio\Data Analyst, Scientiest, ML, DL, AI\Retail Shrinkage & Loss Prevention Analytics\retail_shrinkage_analytics


In [2]:
import pandas as pd
import os


In [3]:
transactions = pd.read_csv('raw_data/transaction_data.csv')
products     = pd.read_csv('raw_data/product.csv')
demographics = pd.read_csv('raw_data/hh_demographic.csv')

## Basic Info

In [4]:
for name, df in [('TRANSACTIONS', transactions), 
                ('PRODUCTS', products), 
                ('DEMOGRAPHICS', demographics)]:
    print(f"\n📋 {name}")
    print(f"   Shape      : {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"   Columns    : {list(df.columns)}")
    print(f"   Null values: {df.isnull().sum().sum()} total")
    print(df.head(3))


📋 TRANSACTIONS
   Shape      : 2,595,732 rows × 12 columns
   Columns    : ['household_key', 'BASKET_ID', 'DAY', 'PRODUCT_ID', 'QUANTITY', 'SALES_VALUE', 'STORE_ID', 'RETAIL_DISC', 'TRANS_TIME', 'WEEK_NO', 'COUPON_DISC', 'COUPON_MATCH_DISC']
   Null values: 0 total
   household_key    BASKET_ID  DAY  PRODUCT_ID  QUANTITY  SALES_VALUE  \
0           2375  26984851472    1     1004906         1         1.39   
1           2375  26984851472    1     1033142         1         0.82   
2           2375  26984851472    1     1036325         1         0.99   

   STORE_ID  RETAIL_DISC  TRANS_TIME  WEEK_NO  COUPON_DISC  COUPON_MATCH_DISC  
0       364         -0.6        1631        1          0.0                0.0  
1       364          0.0        1631        1          0.0                0.0  
2       364         -0.3        1631        1          0.0                0.0  

📋 PRODUCTS
   Shape      : 92,353 rows × 7 columns
   Columns    : ['PRODUCT_ID', 'MANUFACTURER', 'DEPARTMENT', 'BRAND'

In [5]:
demographics.head()

,AGE_DESC,MARITAL_STATUS_CODE,INCOME_DESC,HOMEOWNER_DESC,HH_COMP_DESC,HOUSEHOLD_SIZE_DESC,KID_CATEGORY_DESC,household_key
0,65+,A,35-49K,Homeowner,2 Adults No Kids,2,None/Unknown,1
1,45-54,A,50-74K,Homeowner,2 Adults No Kids,2,None/Unknown,7
2,25-34,U,25-34K,Unknown,2 Adults Kids,3,1,8
3,25-34,U,75-99K,Homeowner,2 Adults Kids,4,2,13
4,45-54,B,50-74K,Homeowner,Single Female,1,None/Unknown,16


## Business Stats

In [6]:
print("\n" + "=" * 50)
print("BUSINESS OVERVIEW")
print("=" * 50)

print(f"\n📅 Date range    : Week {transactions['WEEK_NO'].min()} → {transactions['WEEK_NO'].max()}")
print(f"🏠 Unique households : {transactions['household_key'].nunique():,}")
print(f"🛍️  Unique products   : {transactions['PRODUCT_ID'].nunique():,}")
print(f"🏪 Unique stores     : {transactions['STORE_ID'].nunique():,}")
print(f"💰 Total revenue     : ${transactions['SALES_VALUE'].sum():,.2f}")
print(f"📦 Total quantity    : {transactions['QUANTITY'].sum():,.0f} units")


BUSINESS OVERVIEW

📅 Date range    : Week 1 → 102
🏠 Unique households : 2,500
🛍️  Unique products   : 92,339
🏪 Unique stores     : 582
💰 Total revenue     : $8,057,463.08
📦 Total quantity    : 260,685,622 units


## Lost Function

| Skenario | Yang Terjadi di Data |
| -------- | -------- |
| Kasir batalkan transaksi setelah di-scan | SALES_VALUE negatif   |
| Pelanggan kembalikan barang  | QUANTITY negatif  |
| Admin error / price override | SALES_VALUE tidak wajar |
| Kasir curang — scan lalu void untuk ambil uang | Pola void berulang di kasir yang sama |

In [7]:
neg_sales = transactions[transactions['SALES_VALUE'] < 0]
print(f"\n⚠️  Negative SALES_VALUE  : {len(neg_sales):,} rows ({len(neg_sales)/len(transactions)*100:.2f}%)")

# Quantity negatif
neg_qty = transactions[transactions['QUANTITY'] < 0]
print(f"⚠️  Negative QUANTITY     : {len(neg_qty):,} rows ({len(neg_qty)/len(transactions)*100:.2f}%)")

# Sales value = 0
zero_sales = transactions[transactions['SALES_VALUE'] == 0]
print(f"⚠️  Zero SALES_VALUE      : {len(zero_sales):,} rows ({len(zero_sales)/len(transactions)*100:.2f}%)")


⚠️  Negative SALES_VALUE  : 0 rows (0.00%)
⚠️  Negative QUANTITY     : 0 rows (0.00%)
⚠️  Zero SALES_VALUE      : 18,850 rows (0.73%)


* ✅ QUANTITY negatif      → indikasi return/void

* ✅ SALES_VALUE negatif   → indikasi refund

* ✅ SALES_VALUE = 0       → barang keluar tanpa bayar?

* ✅ Rasio void per kasir  → kasir mana yang sering void?

* ✅ Pola waktu void        → jam berapa void sering terjadi?

* ✅ Kategori produk        → produk mahal lebih sering di-void?


### Zero Sales Value Investigation

In [8]:
zero_sales = transactions[transactions['SALES_VALUE'] == 0]

print("🔍 INVESTIGASI ZERO SALES VALUE")
print(f"Total rows  : {len(zero_sales):,}")
print(f"Unique stores    : {zero_sales['STORE_ID'].nunique()}")
print(f"Unique products  : {zero_sales['PRODUCT_ID'].nunique()}")

print("\n📊 Distribusi QUANTITY pada zero sales:")
print(zero_sales['QUANTITY'].value_counts().head(10))

print("\n💰 Statistik RETAIL_DISC pada zero sales:")
print(zero_sales['RETAIL_DISC'].describe())

print("\n🎟️  Apakah ada COUPON_DISC pada zero sales?")
print(zero_sales['COUPON_DISC'].value_counts().head())

# Bandingkan: zero sales VS normal transactions
print("\n⚖️  PERBANDINGAN RATA-RATA DISKON")
print(f"Normal transactions - avg RETAIL_DISC  : {transactions[transactions['SALES_VALUE'] > 0]['RETAIL_DISC'].mean():.4f}")
print(f"Zero sales          - avg RETAIL_DISC  : {zero_sales['RETAIL_DISC'].mean():.4f}")
print(f"Zero sales          - avg COUPON_DISC  : {zero_sales['COUPON_DISC'].mean():.4f}")

🔍 INVESTIGASI ZERO SALES VALUE
Total rows  : 18,850
Unique stores    : 187
Unique products  : 5330

📊 Distribusi QUANTITY pada zero sales:
QUANTITY
0    14399
1     4370
2       75
3        5
9        1
Name: count, dtype: int64

💰 Statistik RETAIL_DISC pada zero sales:
count    18850.000000
mean        -0.781153
std          1.816268
min        -39.950000
25%          0.000000
50%          0.000000
75%          0.000000
max          0.010000
Name: RETAIL_DISC, dtype: float64

🎟️  Apakah ada COUPON_DISC pada zero sales?
COUPON_DISC
 0.0    13852
-1.0     1780
-2.0      667
-3.0      290
-1.5      288
Name: count, dtype: int64

⚖️  PERBANDINGAN RATA-RATA DISKON
Normal transactions - avg RETAIL_DISC  : -0.5369
Zero sales          - avg RETAIL_DISC  : -0.7812
Zero sales          - avg COUPON_DISC  : -0.5505


**ZERO SALES (18,850 rows)**

* QUANTITY = 0  (14,399) → Ghost Transaction ⚠️
Tidak ada barang, tidak ada nilai = administrative error

* QUANTITY > 0 + COUPON_DISC tinggi (sebagian 4,451) → Free Item via Promo ✅
Barang keluar, nilai = 0 karena diskon 100% = legitimate

* QUANTITY > 0 + tanpa diskon signifikan → Suspicious ⚠️
Barang keluar, nilai = 0, tidak ada alasan jelas = red flag

### Segmentasi Zero Sales 


In [9]:
print("=" * 55)
print("SEGMENTASI ZERO SALES TRANSACTION")
print("=" * 55)

# Segmen 1: Ghost Transaction (qty = 0)
ghost = zero_sales[zero_sales['QUANTITY'] == 0]

# Segmen 2: Free item via promo (qty > 0, ada diskon)
free_promo = zero_sales[
    (zero_sales['QUANTITY'] > 0) & 
    (zero_sales['RETAIL_DISC'] != 0) | (zero_sales['COUPON_DISC'] != 0)
]

# Segmen 3: Suspicious (qty > 0, tidak ada diskon apapun)
suspicious = zero_sales[
    (zero_sales['QUANTITY'] > 0) & 
    (zero_sales['RETAIL_DISC'] == 0) & 
    (zero_sales['COUPON_DISC'] == 0) &
    (zero_sales['COUPON_MATCH_DISC'] == 0)
]

print(f"\n👻 Segmen 1 - Ghost Transaction    : {len(ghost):,} rows")
print(f"🎁 Segmen 2 - Free Item via Promo  : {len(free_promo):,} rows")
print(f"🚨 Segmen 3 - Suspicious (no disc) : {len(suspicious):,} rows")

# Detail Segmen 3 — ini yang paling menarik
print("\n🔎 Detail Suspicious Transactions:")
print(f"   Unique stores   : {suspicious['STORE_ID'].nunique()}")
print(f"   Unique products : {suspicious['PRODUCT_ID'].nunique()}")
print(f"   Total quantity  : {suspicious['QUANTITY'].sum():,} units keluar tanpa bayar")

print("\n📅 Distribusi per DAY (suspicious):")
print(suspicious['DAY'].value_counts().head(10))

print("\n🏪 Top 10 stores dengan suspicious transactions:")
print(suspicious['STORE_ID'].value_counts().head(10))

SEGMENTASI ZERO SALES TRANSACTION

👻 Segmen 1 - Ghost Transaction    : 14,399 rows
🎁 Segmen 2 - Free Item via Promo  : 8,704 rows
🚨 Segmen 3 - Suspicious (no disc) : 702 rows

🔎 Detail Suspicious Transactions:
   Unique stores   : 116
   Unique products : 59
   Total quantity  : 732 units keluar tanpa bayar

📅 Distribusi per DAY (suspicious):
DAY
183    9
191    7
165    6
242    6
102    6
65     5
160    5
200    5
401    5
225    5
Name: count, dtype: int64

🏪 Top 10 stores dengan suspicious transactions:
STORE_ID
316      67
330      26
318      23
343      22
384      18
432      17
427      17
372      15
32004    15
374      14
Name: count, dtype: int64


⚠️ Segmen 2 overlap dengan Segmen 1 — ada ghost transaction yang juga punya diskon. Wajar, logika filter kita memang bisa overlap. Untuk project ini kita fokus ke 702 suspicious rows.

59 produk dari 116 toko — artinya ada produk tertentu yang konsisten muncul dalam suspicious transactions. Ini lebih mengarah ke administrative error sistemik atau produk yang sengaja di-void berulang.

| Skenario | Legitimate? |
| -------- | -------- |
| Produk rusak dicatat keluar tanpa nilai | ✅ Legitimate — waste/spoilage   |
| Sample gratis untuk promosi  | ✅ Legitimate — marketing cost  |
| Kasir lupa input harga | ⚠️ Admin error |
| Kasir sengaja tidak charge teman/keluarga | ❌ Internal theft |
| Void setelah barang sudah dibawa keluar | ❌ Fraud |

Data saja tidak cukup untuk membuktikan kecurangan. Data hanya bisa menunjukkan anomali — investigasi lapangan tetap diperlukan. Tugas kita sebagai analyst adalah mempersempit area investigasi dari 2.5 juta transaksi menjadi 702 baris yang perlu dicek.

# Business Question Answer

## Produk paling sering di Suspicious Transactions ───────

In [10]:

print("🏷️  TOP 20 PRODUCTS dalam Suspicious Transactions")
print("=" * 55)

# Merge dengan product data untuk lihat nama kategori
top_products = suspicious.groupby('PRODUCT_ID').agg(
    total_quantity = ('QUANTITY', 'sum'),
    total_occurence = ('PRODUCT_ID', 'count'),
    unique_stores = ('STORE_ID', 'nunique')
).sort_values('total_occurence', ascending=False).head(20)

# Join dengan product info
top_products = top_products.merge(
    products[['PRODUCT_ID', 'DEPARTMENT', 'COMMODITY_DESC']],
    on='PRODUCT_ID',
    how='left'
)

print(top_products.to_string())

print("\n📦 Departemen paling banyak di Suspicious Transactions:")
dept_suspicious = suspicious.merge(
    products[['PRODUCT_ID', 'DEPARTMENT']], 
    on='PRODUCT_ID', how='left'
)
print(dept_suspicious['DEPARTMENT'].value_counts().head(10))

🏷️  TOP 20 PRODUCTS dalam Suspicious Transactions
    PRODUCT_ID  total_quantity  total_occurence  unique_stores      DEPARTMENT       COMMODITY_DESC
0      1053460             331              323             38  COUP/STR & MFG  COUPONS/STORE & MFG
1       897752              54               54             34         GROCERY           BAG SNACKS
2      1079228              46               46             33         GROCERY           BAG SNACKS
3      1013587              42               42              9  COUP/STR & MFG  COUPONS/STORE & MFG
4       991268              39               38             20         GROCERY           BAG SNACKS
5      5574954              31               29             23  COUP/STR & MFG  COUPONS/STORE & MFG
6       885430              20               20             14         GROCERY           BAG SNACKS
7       744587              19               19              5  COUP/STR & MFG  COUPONS/STORE & MFG
8      1049131              17               16   

Dua Departemen Dominan:

* COUP/STR & MFG  → Coupon/Store & Manufacturer coupons
* GROCERY         → Khususnya BAG SNACKS

Produk COUP/STR & MFG dengan semua diskon = 0 tapi SALES_VALUE = 0 mengindikasikan:
* Kemungkinan 1: Kupon di-redeem tapi tidak tercatat di kolom diskon → Administrative/system error

* Kemungkinan 2: Kasir manual override ke $0 tanpa kupon valid → Potensi internal fraud

💡 Insight kunci: PRODUCT_ID 1053460 muncul di 38 toko berbeda dengan 323 kejadian — ini bukan kebetulan. Pola yang tersebar luas di banyak toko mengarah ke system/administrative issue, bukan fraud individu.